# TMDB Movie Data Pipeline & Ingestion

## Environment & Authentication

In [ ]:
import getpass
import sqlite3
import requests

access_token = getpass.getpass("Enter your TMDB API access token: ")

session = requests.Session()
session.headers.update({
    "accept": "application/json",
    "Authorization": f"Bearer {access_token}",
})

print("TMDB session configured.")

## Extract: Popular Movies

In [ ]:
all_raw_movies = []

for page in range(1, 6):
    url = "https://api.themoviedb.org/3/movie/popular"
    response = session.get(url, params={"page": page}, timeout=30)
    response.raise_for_status()

    payload = response.json()
    all_raw_movies.extend(payload.get("results", []))

print(f"Retrieved {len(all_raw_movies)} movies across 5 pages.")

## Transform: Enrich Movie Details

The `/movie/popular` endpoint includes movie IDs and genre IDs, but not runtime or full genre names. For each movie, this step requests `/movie/{id}` to collect those fields before loading the records into SQLite.

In [ ]:
movie_records = []
failed_movie_ids = []

for movie in all_raw_movies:
    movie_id = movie.get("id")
    if movie_id is None:
        continue

    try:
        detail_url = f"https://api.themoviedb.org/3/movie/{movie_id}"
        response = session.get(detail_url, timeout=30)
        response.raise_for_status()
        details = response.json()
    except requests.RequestException as exc:
        failed_movie_ids.append(movie_id)
        print(f"Skipping movie {movie_id}: {exc}")
        continue

    genres = ", ".join(
        genre["name"]
        for genre in details.get("genres", [])
        if genre.get("name")
    )

    movie_records.append({
        "tmdb_id": movie_id,
        "title": details.get("title"),
        "runtime_minutes": details.get("runtime"),
        "release_date": details.get("release_date") or None,
        "genres": genres,
    })

print(f"Prepared {len(movie_records)} records for loading.")
if failed_movie_ids:
    print(f"Skipped {len(failed_movie_ids)} movies because detail requests failed.")

## Load: SQLite

In [ ]:
if not movie_records:
    raise RuntimeError("No movie records were prepared. Check your TMDB token and API responses before loading the database.")

with sqlite3.connect("movies.db") as connection:
    cursor = connection.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS movies (
        tmdb_id INTEGER PRIMARY KEY,
        title TEXT NOT NULL,
        runtime_minutes INTEGER,
        release_date TEXT,
        genres TEXT
    );
    """)

    cursor.execute("DELETE FROM movies;")

    cursor.executemany("""
        INSERT INTO movies (tmdb_id, title, runtime_minutes, release_date, genres)
        VALUES (:tmdb_id, :title, :runtime_minutes, :release_date, :genres)
    """, movie_records)

print(f"Loaded {len(movie_records)} records into movies.db.")

## Audit & Integrity Verification

In [ ]:
with sqlite3.connect("movies.db") as connection:
    cursor = connection.cursor()
    cursor.execute("SELECT COUNT(*) FROM movies;")
    row_count = cursor.fetchone()[0]

    cursor.execute("""
        SELECT tmdb_id, title, runtime_minutes, release_date, genres
        FROM movies
        ORDER BY title
        LIMIT 5;
    """)
    sample_rows = cursor.fetchall()

print(f"Database audit: {row_count} records found.")
for row in sample_rows:
    print(row)